# Notebook 2: Generative AI Concepts — From DDPM to Materials

**APS Tutorial T4 — Generative AI for Physics: From Models to Materials**

This notebook builds intuition for:
- What generative models do (and don't do)
- How **diffusion models** (DDPM) work — with physics analogies
- **Hands-on: train a DDPM on MNIST** and generate handwritten digits
- **Conditional generation** — generating specific digit classes
- Why diffusion models are well-suited for materials structure prediction

> This notebook trains a small diffusion model on MNIST (~5 min on GPU).
> Make sure you have a GPU runtime enabled.

In [ ]:
# --- Setup ---
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'scipy'])

import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
print(f'PyTorch: {torch.__version__}')
if device == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
print('Ready.')

---
## 2.1 Discriminative vs. generative models

| | Discriminative | Generative |
|---|---|---|
| **Task** | Predict a label from data | Create new data samples |
| **Learns** | P(label \| data) | P(data) |
| **Materials example** | "Is this structure stable?" | "Generate a new stable structure" |

A generative model learns the **probability distribution** over its training data,
then samples from that distribution to create new examples.

**Physics analogy:** Think of the training data as defining an energy landscape.
The generative model learns this landscape, and sampling is like finding new
low-energy configurations.

---
## 2.2 A brief taxonomy of generative models

| Model | Key idea | Physics analogy |
|-------|----------|-----------------|
| **VAE** | Encode → latent space → decode | Compressed representation of configuration space |
| **GAN** | Generator vs. discriminator game | Adversarial optimization |
| **Flow** | Invertible transformations | Canonical transformations in phase space |
| **Diffusion** | Gradually add then remove noise | Heating → controlled cooling (annealing) |

**Diffusion models** have become the dominant approach for crystal generation
because they naturally handle the periodicity and symmetry of crystal structures.

---
## 2.3 How diffusion models work

### The forward process: adding noise (DDPM)

A **Denoising Diffusion Probabilistic Model** (DDPM) defines a Markov chain that
gradually corrupts data $\mathbf{x}_0$ into pure Gaussian noise over $T$ steps.

At each step $t$, a small amount of noise is added:

$$q(\mathbf{x}_t \mid \mathbf{x}_{t-1}) = \mathcal{N}(\mathbf{x}_t;\, \sqrt{1-\beta_t}\,\mathbf{x}_{t-1},\; \beta_t \mathbf{I})$$

where $\beta_t \in (0, 1)$ is the **noise schedule** — a small number that controls
how much noise is added at step $t$.

Thanks to the reparameterization trick, we can jump directly to any step $t$ without
iterating:

$$q(\mathbf{x}_t \mid \mathbf{x}_0) = \mathcal{N}(\mathbf{x}_t;\, \sqrt{\bar{\alpha}_t}\,\mathbf{x}_0,\; (1-\bar{\alpha}_t)\mathbf{I})$$

where $\alpha_t = 1 - \beta_t$ and $\bar{\alpha}_t = \prod_{s=1}^{t} \alpha_s$ is the
cumulative signal retention. As $t \to T$, $\bar{\alpha}_T \approx 0$ and the data
becomes pure noise.

**Physics analogy:** This is like **heating a crystal until it melts** into a
disordered liquid. $\beta_t$ is the heating rate, and $\bar{\alpha}_t$ tracks
how much crystalline order remains.

### The reverse process: denoising

The reverse process learns to undo the noising, generating data from noise:

$$p_\theta(\mathbf{x}_{t-1} \mid \mathbf{x}_t) = \mathcal{N}(\mathbf{x}_{t-1};\, \boldsymbol{\mu}_\theta(\mathbf{x}_t, t),\; \sigma_t^2 \mathbf{I})$$

A neural network $\boldsymbol{\epsilon}_\theta(\mathbf{x}_t, t)$ is trained to predict the noise
$\boldsymbol{\epsilon}$ that was added, minimizing the simple loss:

$$\mathcal{L} = \mathbb{E}_{t, \mathbf{x}_0, \boldsymbol{\epsilon}}\left[\|\boldsymbol{\epsilon} - \boldsymbol{\epsilon}_\theta(\mathbf{x}_t, t)\|^2\right]$$

Once trained, the predicted mean is:

$$\boldsymbol{\mu}_\theta(\mathbf{x}_t, t) = \frac{1}{\sqrt{\alpha_t}}\left(\mathbf{x}_t - \frac{\beta_t}{\sqrt{1-\bar{\alpha}_t}}\,\boldsymbol{\epsilon}_\theta(\mathbf{x}_t, t)\right)$$

**Physics analogy:** This is like **controlled crystallization from a melt**.
The model has learned the "crystallization dynamics" and guides random atoms
into a coherent crystal.

### The score function perspective

Equivalently, the network learns the **score function** $\nabla_{\mathbf{x}} \log p(\mathbf{x})$ —
the gradient of the log-probability density. In plain terms: "which direction should
I move each atom to make this look more like a real crystal?"

The noise prediction $\boldsymbol{\epsilon}_\theta$ and score $\mathbf{s}_\theta$ are related by:

$$\mathbf{s}_\theta(\mathbf{x}_t, t) = -\frac{\boldsymbol{\epsilon}_\theta(\mathbf{x}_t, t)}{\sqrt{1 - \bar{\alpha}_t}}$$

In the next sections, we'll see DDPM in action — first on a 2D toy example,
then training a real model on MNIST handwritten digits.

### 2.4 Toy demo: the forward process

The plot below shows data points arranged in a ring of 6 clusters (like atoms at
lattice sites). As we increase the noise level $\sigma$, the points diffuse outward
until the original structure is lost — analogous to melting a crystal.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm

np.random.seed(42)

# Define a simple 2D target distribution (mixture of Gaussians arranged in a ring)
def target_samples(n=500):
    """Generate samples from a ring of Gaussians (like crystal sites)."""
    n_centers = 6
    angles = np.linspace(0, 2*np.pi, n_centers, endpoint=False)
    centers = np.stack([2*np.cos(angles), 2*np.sin(angles)], axis=1)
    samples = []
    for _ in range(n):
        c = centers[np.random.randint(n_centers)]
        samples.append(c + 0.15 * np.random.randn(2))
    return np.array(samples)

# Simulate the forward (noising) process
data = target_samples(500)
noise_levels = [0.0, 0.3, 0.8, 2.0]

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, sigma in zip(axes, noise_levels):
    noised = data + sigma * np.random.randn(*data.shape)
    ax.scatter(noised[:, 0], noised[:, 1], s=8, alpha=0.5, c='steelblue')
    ax.set_xlim(-5, 5)
    ax.set_ylim(-5, 5)
    ax.set_aspect('equal')
    if sigma == 0:
        ax.set_title('Original data\n(crystal sites)', fontsize=11)
    else:
        ax.set_title(f'Noise level σ = {sigma}', fontsize=11)
    ax.set_xticks([])
    ax.set_yticks([])

fig.suptitle('Forward process: gradually adding noise ("heating")', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

### Toy demo: the reverse (denoising) process

Now we simulate the **reverse process**: starting from pure noise, we iteratively
denoise toward the target distribution. Our toy denoiser estimates the score function
by finding the direction toward the nearest data clusters:

$$\hat{\mathbf{s}}(\mathbf{x}) \approx \frac{1}{k}\sum_{i \in \text{kNN}(\mathbf{x})} (\mathbf{x}_i - \mathbf{x})$$

At each step, we move points in the score direction and add a small amount of noise
(the stochastic part of Langevin dynamics):

$$\mathbf{x}_{t-1} = \mathbf{x}_t + \eta\,\hat{\mathbf{s}}(\mathbf{x}_t) + \sqrt{2\eta}\,\boldsymbol{\xi}, \quad \boldsymbol{\xi} \sim \mathcal{N}(0, \mathbf{I})$$

Watch how the disordered "melt" gradually crystallizes back into the ring pattern.

In [ ]:
# Simulate the reverse (denoising) process
# Starting from noise, gradually denoise toward the target distribution

def simple_denoise_step(x, data, sigma_current, sigma_next, n_neighbors=10):
    """A toy denoising step: move points toward nearest cluster centers."""
    from scipy.spatial.distance import cdist
    # Estimate score: direction toward nearest data points
    dists = cdist(x, data)
    nearest_idx = np.argsort(dists, axis=1)[:, :n_neighbors]
    target = np.mean(data[nearest_idx], axis=1)
    # Move partway toward target
    step_size = 1 - sigma_next / sigma_current
    x_new = x + step_size * (target - x)
    # Add small noise (except at final step)
    if sigma_next > 0:
        x_new += sigma_next * 0.3 * np.random.randn(*x.shape)
    return x_new

# Start from pure noise
x = 3.5 * np.random.randn(500, 2)
sigmas = [2.0, 0.8, 0.3, 0.0]

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
axes[0].scatter(x[:, 0], x[:, 1], s=8, alpha=0.5, c='coral')
axes[0].set_title('Pure noise\n("disordered melt")', fontsize=11)
axes[0].set_xlim(-5, 5); axes[0].set_ylim(-5, 5)
axes[0].set_aspect('equal'); axes[0].set_xticks([]); axes[0].set_yticks([])

for i in range(1, 4):
    x = simple_denoise_step(x, data, sigmas[i-1], sigmas[i] if i < 3 else 0)
    color = ['coral', 'mediumpurple', 'steelblue'][i-1]
    axes[i].scatter(x[:, 0], x[:, 1], s=8, alpha=0.5, c=color)
    if i < 3:
        axes[i].set_title(f'Denoising step {i}', fontsize=11)
    else:
        axes[i].set_title('Generated samples\n("crystallized")', fontsize=11)
    axes[i].set_xlim(-5, 5); axes[i].set_ylim(-5, 5)
    axes[i].set_aspect('equal'); axes[i].set_xticks([]); axes[i].set_yticks([])

fig.suptitle('Reverse process: denoising ("controlled crystallization")', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

### Animated view

Here's the same forward → reverse process as an animation:

In [ ]:
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

fig_anim, ax_anim = plt.subplots(figsize=(5, 5))

# Forward then reverse: 20 frames
n_frames = 20
sigmas_fwd = np.linspace(0, 2.5, n_frames // 2)
sigmas_rev = sigmas_fwd[::-1]
all_sigmas = np.concatenate([sigmas_fwd, sigmas_rev])

def animate(frame):
    ax_anim.clear()
    sigma = all_sigmas[frame]
    if frame < n_frames // 2:
        noised = data + sigma * np.random.randn(*data.shape)
        color = 'steelblue'
        phase = f'Forward (σ={sigma:.1f})'
    else:
        # Simple denoising approximation
        noise_level = sigma
        noised = data + noise_level * np.random.randn(*data.shape)
        color = '#E69F00'
        phase = f'Reverse (σ={sigma:.1f})'
    ax_anim.scatter(noised[:, 0], noised[:, 1], s=8, alpha=0.5, c=color)
    ax_anim.set_xlim(-5, 5)
    ax_anim.set_ylim(-5, 5)
    ax_anim.set_aspect('equal')
    ax_anim.set_xticks([])
    ax_anim.set_yticks([])
    ax_anim.set_title(phase, fontsize=12)

anim = FuncAnimation(fig_anim, animate, frames=n_frames, interval=300)
plt.close(fig_anim)
HTML(anim.to_jshtml())

### The noise schedule

The noise schedule $\{\beta_t\}_{t=1}^T$ controls the rate of corruption. A **linear
schedule** starts with small $\beta_1 \approx 10^{-4}$ and increases to $\beta_T \approx 0.02$.

The cumulative product $\bar{\alpha}_t = \prod_{s=1}^t (1 - \beta_s)$ tells us how
much of the original signal survives at step $t$. When $\bar{\alpha}_t \approx 0$,
the data is essentially pure noise.

In [ ]:
# Noise schedule visualization
T = 1000  # total diffusion steps
betas = np.linspace(1e-4, 0.02, T)  # linear schedule
alphas = 1 - betas
alpha_bars = np.cumprod(alphas)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(range(T), betas, color='steelblue', linewidth=1.5)
axes[0].set_xlabel('Diffusion step t')
axes[0].set_ylabel('β_t (noise added per step)')
axes[0].set_title('Noise schedule β_t')

axes[1].plot(range(T), alpha_bars, color='#E69F00', linewidth=1.5)
axes[1].set_xlabel('Diffusion step t')
axes[1].set_ylabel('ᾱ_t (signal remaining)')
axes[1].set_title('Cumulative signal retention ᾱ_t')
axes[1].axhline(y=0.5, color='gray', linestyle='--', alpha=0.5, label='50% signal')
axes[1].legend()

plt.tight_layout()
plt.show()

print('At t=0: structure is clean (ᾱ ≈ 1)')
print(f'At t={T}: structure is nearly pure noise (ᾱ = {alpha_bars[-1]:.6f})')

The toy example above shows the core idea:
- The **forward process** destroys structure by adding noise
- The **reverse process** creates structure by removing noise
- The neural network learns to perform the reverse process

In a real crystal diffusion model (like SCIGEN), the same principle applies
but in 3D, with periodic boundary conditions, and the "data" includes:
- Atom positions (fractional coordinates)
- Atom types (element species)
- Lattice parameters (the unit cell shape)

---
## 2.5 Hands-on: training a DDPM on MNIST

Let's put the theory into practice. We'll train a small diffusion model to generate
handwritten digits from the MNIST dataset. This uses the same DDPM algorithm that
underlies crystal structure generation — just applied to 28x28 images instead of
atom positions.

The implementation below is adapted from
[TeaPearce/Conditional_Diffusion_MNIST](https://github.com/TeaPearce/Conditional_Diffusion_MNIST)
(MIT License), which implements **classifier-free diffusion guidance** for
class-conditional generation.

### Model architecture

- **UNet** with residual convolutional blocks
- **Time embedding**: sinusoidal encoding of the diffusion step $t$
- **Class embedding**: one-hot digit label (0–9) for conditional generation
- **Classifier-free guidance**: randomly drop the class label during training
  (with probability 0.1), so the model learns both conditional and unconditional generation

In [ ]:
# === DDPM Model Definition ===
# Adapted from TeaPearce/Conditional_Diffusion_MNIST (MIT License)
# https://github.com/TeaPearce/Conditional_Diffusion_MNIST

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import transforms
from torchvision.datasets import MNIST
from torchvision.utils import make_grid
import numpy as np
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

class ResidualConvBlock(nn.Module):
    def __init__(self, in_channels, out_channels, is_res=False):
        super().__init__()
        self.same_channels = in_channels == out_channels
        self.is_res = is_res
        self.conv1 = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 3, 1, 1),
            nn.BatchNorm2d(out_channels), nn.GELU())
        self.conv2 = nn.Sequential(
            nn.Conv2d(out_channels, out_channels, 3, 1, 1),
            nn.BatchNorm2d(out_channels), nn.GELU())

    def forward(self, x):
        if self.is_res:
            x1 = self.conv1(x)
            x2 = self.conv2(x1)
            out = (x + x2) if self.same_channels else (x1 + x2)
            return out / 1.414
        return self.conv2(self.conv1(x))

class UnetDown(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.model = nn.Sequential(ResidualConvBlock(in_ch, out_ch), nn.MaxPool2d(2))
    def forward(self, x):
        return self.model(x)

class UnetUp(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.model = nn.Sequential(
            nn.ConvTranspose2d(in_ch, out_ch, 2, 2),
            ResidualConvBlock(out_ch, out_ch),
            ResidualConvBlock(out_ch, out_ch))
    def forward(self, x, skip):
        return self.model(torch.cat((x, skip), 1))

class EmbedFC(nn.Module):
    def __init__(self, input_dim, emb_dim):
        super().__init__()
        self.input_dim = input_dim
        self.model = nn.Sequential(
            nn.Linear(input_dim, emb_dim), nn.GELU(), nn.Linear(emb_dim, emb_dim))
    def forward(self, x):
        return self.model(x.view(-1, self.input_dim))

class ContextUnet(nn.Module):
    """UNet with time and class conditioning for DDPM."""
    def __init__(self, in_channels, n_feat=128, n_classes=10):
        super().__init__()
        self.in_channels = in_channels
        self.n_feat = n_feat
        self.n_classes = n_classes

        self.init_conv = ResidualConvBlock(in_channels, n_feat, is_res=True)
        self.down1 = UnetDown(n_feat, n_feat)
        self.down2 = UnetDown(n_feat, 2 * n_feat)
        self.to_vec = nn.Sequential(nn.AvgPool2d(7), nn.GELU())

        self.timeembed1 = EmbedFC(1, 2 * n_feat)
        self.timeembed2 = EmbedFC(1, n_feat)
        self.contextembed1 = EmbedFC(n_classes, 2 * n_feat)
        self.contextembed2 = EmbedFC(n_classes, n_feat)

        self.up0 = nn.Sequential(
            nn.ConvTranspose2d(2 * n_feat, 2 * n_feat, 7, 7),
            nn.GroupNorm(8, 2 * n_feat), nn.ReLU())
        self.up1 = UnetUp(4 * n_feat, n_feat)
        self.up2 = UnetUp(2 * n_feat, n_feat)
        self.out = nn.Sequential(
            nn.Conv2d(2 * n_feat, n_feat, 3, 1, 1),
            nn.GroupNorm(8, n_feat), nn.ReLU(),
            nn.Conv2d(n_feat, self.in_channels, 3, 1, 1))

    def forward(self, x, c, t, context_mask):
        x = self.init_conv(x)
        down1 = self.down1(x)
        down2 = self.down2(down1)
        hiddenvec = self.to_vec(down2)

        c = F.one_hot(c, num_classes=self.n_classes).float()
        context_mask = (-1 * (1 - context_mask[:, None].repeat(1, self.n_classes)))
        c = c * context_mask

        cemb1 = self.contextembed1(c).view(-1, self.n_feat * 2, 1, 1)
        temb1 = self.timeembed1(t).view(-1, self.n_feat * 2, 1, 1)
        cemb2 = self.contextembed2(c).view(-1, self.n_feat, 1, 1)
        temb2 = self.timeembed2(t).view(-1, self.n_feat, 1, 1)

        up1 = self.up0(hiddenvec)
        up2 = self.up1(cemb1 * up1 + temb1, down2)
        up3 = self.up2(cemb2 * up2 + temb2, down1)
        return self.out(torch.cat((up3, x), 1))

def ddpm_schedules(beta1, beta2, T):
    """Pre-compute DDPM noise schedule quantities."""
    beta_t = (beta2 - beta1) * torch.arange(0, T + 1, dtype=torch.float32) / T + beta1
    alpha_t = 1 - beta_t
    alphabar_t = torch.cumsum(torch.log(alpha_t), dim=0).exp()
    return {
        'alpha_t': alpha_t,
        'oneover_sqrta': 1 / torch.sqrt(alpha_t),
        'sqrt_beta_t': torch.sqrt(beta_t),
        'alphabar_t': alphabar_t,
        'sqrtab': torch.sqrt(alphabar_t),
        'sqrtmab': torch.sqrt(1 - alphabar_t),
        'mab_over_sqrtmab': (1 - alpha_t) / torch.sqrt(1 - alphabar_t),
    }

class DDPM(nn.Module):
    def __init__(self, nn_model, betas, n_T, device, drop_prob=0.1):
        super().__init__()
        self.nn_model = nn_model.to(device)
        for k, v in ddpm_schedules(betas[0], betas[1], n_T).items():
            self.register_buffer(k, v)
        self.n_T = n_T
        self.device = device
        self.drop_prob = drop_prob
        self.loss_mse = nn.MSELoss()

    def forward(self, x, c):
        """Training step: add noise at random t, predict it."""
        _ts = torch.randint(1, self.n_T + 1, (x.shape[0],)).to(self.device)
        noise = torch.randn_like(x)
        x_t = self.sqrtab[_ts, None, None, None] * x + self.sqrtmab[_ts, None, None, None] * noise
        context_mask = torch.bernoulli(torch.zeros_like(c, dtype=torch.float32) + self.drop_prob).to(self.device)
        return self.loss_mse(noise, self.nn_model(x_t, c, _ts / self.n_T, context_mask))

    def sample(self, n_sample, size, device, guide_w=0.0):
        """Generate samples using classifier-free guidance."""
        x_i = torch.randn(n_sample, *size).to(device)
        c_i = torch.arange(0, 10).to(device).repeat(int(n_sample / 10))
        context_mask = torch.zeros_like(c_i, dtype=torch.float32).to(device)

        c_i = c_i.repeat(2)
        context_mask = context_mask.repeat(2)
        context_mask[n_sample:] = 1.

        x_i_store = []
        for i in range(self.n_T, 0, -1):
            t_is = torch.tensor([i / self.n_T]).to(device).repeat(n_sample, 1, 1, 1)
            x_i = x_i.repeat(2, 1, 1, 1)
            t_is = t_is.repeat(2, 1, 1, 1)

            z = torch.randn(n_sample, *size).to(device) if i > 1 else 0
            eps = self.nn_model(x_i, c_i, t_is, context_mask)
            eps1, eps2 = eps[:n_sample], eps[n_sample:]
            eps = (1 + guide_w) * eps1 - guide_w * eps2
            x_i = x_i[:n_sample]
            x_i = self.oneover_sqrta[i] * (x_i - eps * self.mab_over_sqrtmab[i]) + self.sqrt_beta_t[i] * z

            if i % 50 == 0 or i == self.n_T or i < 8:
                x_i_store.append(x_i.detach().cpu().numpy())

        return x_i, np.array(x_i_store)

print('Model classes defined.')
n_params = sum(p.numel() for p in ContextUnet(1, 128, 10).parameters())
print(f'ContextUnet parameters: {n_params:,}')

### Quick training demo (2 epochs)

We run **just 2 epochs** to see the loss decrease and verify the training loop works.
The results after 2 epochs will be blurry — that's expected! Real generation quality
requires ~20 epochs.

For the generation demo below, we'll load a **pretrained checkpoint** (20 epochs).

In [ ]:
# === Quick training demo (2 epochs) ===
n_epoch = 2        # Short demo — just to see the loss decrease
batch_size = 256
n_T = 400
n_feat = 128
lrate = 1e-4

ddpm = DDPM(
    nn_model=ContextUnet(in_channels=1, n_feat=n_feat, n_classes=10),
    betas=(1e-4, 0.02), n_T=n_T, device=device, drop_prob=0.1
).to(device)

tf = transforms.Compose([transforms.ToTensor()])
dataset = MNIST('./data', train=True, download=True, transform=tf)
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True, num_workers=0)
optim = torch.optim.Adam(ddpm.parameters(), lr=lrate)

loss_history = []
print(f'Training DDPM on MNIST ({n_epoch} epochs, T={n_T})...')
print('(This is a quick demo — pretrained weights are loaded below for generation)')

for ep in range(n_epoch):
    ddpm.train()
    optim.param_groups[0]['lr'] = lrate * (1 - ep / max(n_epoch, 1))
    
    pbar = tqdm(dataloader, desc=f'Epoch {ep+1}/{n_epoch}')
    loss_ema = None
    for x, c in pbar:
        optim.zero_grad()
        loss = ddpm(x.to(device), c.to(device))
        loss.backward()
        optim.step()
        if loss_ema is None:
            loss_ema = loss.item()
        else:
            loss_ema = 0.95 * loss_ema + 0.05 * loss.item()
        pbar.set_description(f'Epoch {ep+1}/{n_epoch} | loss: {loss_ema:.4f}')
    loss_history.append(loss_ema)

fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(range(1, n_epoch + 1), loss_history, 'o-', color='steelblue', linewidth=2)
ax.set_xlabel('Epoch'); ax.set_ylabel('Loss (EMA)')
ax.set_title('DDPM Training Loss (quick demo)', fontsize=13)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Show early results (will be blurry after only 2 epochs)
ddpm.eval()
with torch.no_grad():
    x_gen_early, _ = ddpm.sample(20, (1, 28, 28), device, guide_w=2.0)
grid = make_grid(x_gen_early.clamp(0, 1), nrow=10, padding=1).permute(1, 2, 0).cpu().numpy()
fig, ax = plt.subplots(figsize=(10, 1.5))
ax.imshow(grid, cmap='gray'); ax.axis('off')
ax.set_title(f'After {n_epoch} epochs (blurry — training not converged yet)', fontsize=11)
plt.tight_layout()
plt.show()

### Load pretrained model and generate digits

The 2-epoch demo above shows the training loop works, but the results are blurry.
For high-quality generation, we load a **pretrained checkpoint** (20 epochs).

We then generate digits with **classifier-free guidance** at different strengths:
- $w = 0$: unconditional (no class guidance)
- $w = 0.5$: mild guidance
- $w = 2.0$: strong guidance (sharper, more class-specific digits)

In [ ]:
# === Load pretrained model (20 epochs) ===
import os
from urllib.request import urlretrieve

# Pretrained checkpoint hosted in the GitHub repo (Git LFS)
CKPT_URL = 'https://media.githubusercontent.com/media/RyotaroOKabe/APS_demo_SCIGEN/feat/nb02-mnist-ddpm/models/mnist_ddpm/ddpm_mnist_20ep.pth'
CKPT_PATH = './models/ddpm_mnist_20ep.pth'

# On Colab with cloned repo, use local file if available
PROJECT_DIR = os.environ.get('PROJECT_ROOT', '/content/APS_demo_SCIGEN')
local_ckpt = os.path.join(PROJECT_DIR, 'models', 'mnist_ddpm', 'ddpm_mnist_20ep.pth')
if os.path.exists(local_ckpt):
    CKPT_PATH = local_ckpt
    print(f'Using local checkpoint: {CKPT_PATH}')
elif not os.path.exists(CKPT_PATH):
    os.makedirs('./models', exist_ok=True)
    print('Downloading pretrained DDPM checkpoint (~26 MB)...')
    urlretrieve(CKPT_URL, CKPT_PATH)
    print(f'Saved to {CKPT_PATH}')
else:
    print(f'Using cached checkpoint: {CKPT_PATH}')

# Re-initialize model and load weights
ddpm_pretrained = DDPM(
    nn_model=ContextUnet(in_channels=1, n_feat=128, n_classes=10),
    betas=(1e-4, 0.02), n_T=400, device=device, drop_prob=0.1
).to(device)

try:
    ckpt = torch.load(CKPT_PATH, map_location=device, weights_only=False)
    # Handle both old (raw state_dict) and new (dict with metadata) formats
    if isinstance(ckpt, dict) and 'model_state_dict' in ckpt:
        ddpm_pretrained.load_state_dict(ckpt['model_state_dict'])
        pretrained_loss = ckpt.get('loss_history', None)
        pretrained_epochs = ckpt.get('n_epoch', None)
    else:
        ddpm_pretrained.load_state_dict(ckpt)
        pretrained_loss = None
        pretrained_epochs = None
    print(f'Pretrained model loaded ({pretrained_epochs or "?"} epochs).')
except Exception as e:
    print(f'Could not load pretrained weights: {e}')
    print('Falling back to the 2-epoch model from training above.')
    ddpm_pretrained = ddpm
    pretrained_loss = None

# Plot pretrained training loss curve (if available)
if pretrained_loss is not None:
    fig, ax = plt.subplots(figsize=(8, 3))
    epochs_range = range(1, len(pretrained_loss) + 1)
    ax.plot(epochs_range, pretrained_loss, 'o-', color='coral', linewidth=2, label='Pretrained (full)')
    if loss_history:
        ax.plot(range(1, len(loss_history) + 1), loss_history, 's--', color='steelblue',
                linewidth=2, label=f'Demo ({len(loss_history)} epochs)')
    ax.set_xlabel('Epoch', fontsize=12)
    ax.set_ylabel('Loss (EMA)', fontsize=12)
    ax.set_title('DDPM Training Loss: demo vs pretrained', fontsize=13)
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    print(f'Pretrained final loss: {pretrained_loss[-1]:.4f}')

# === Generate digits with different guidance strengths ===
ddpm_pretrained.eval()
n_sample = 40  # 4 samples per digit class (0-9)
ws = [0.0, 0.5, 2.0]

fig, axes = plt.subplots(len(ws), 1, figsize=(12, 4 * len(ws)))

for ax, w in zip(axes, ws):
    with torch.no_grad():
        x_gen, x_gen_store = ddpm_pretrained.sample(n_sample, (1, 28, 28), device, guide_w=w)
    
    grid = make_grid(x_gen.clamp(0, 1), nrow=10, padding=1).permute(1, 2, 0).cpu().numpy()
    ax.imshow(grid, cmap='gray')
    ax.set_title(f'Guidance w = {w}  (columns = digit classes 0-9)', fontsize=12)
    ax.axis('off')

plt.suptitle('Generated MNIST digits (pretrained, 20 epochs)', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

print('w=0.0: unconditional — diverse but sometimes ambiguous')
print('w=0.5: mild guidance — recognizable digits with some variety')
print('w=2.0: strong guidance — sharp, class-specific digits')

### Visualizing the denoising trajectory

Let's watch a digit emerge from noise step by step — this is the reverse diffusion
process in action.

In [ ]:
# === Denoising trajectory (using pretrained model) ===
with torch.no_grad():
    x_gen, x_gen_store = ddpm_pretrained.sample(40, (1, 28, 28), device, guide_w=2.0)

n_snapshots = x_gen_store.shape[0]
n_show = min(10, n_snapshots)
snapshot_idx = np.linspace(0, n_snapshots - 1, n_show, dtype=int)

fig, axes = plt.subplots(n_show, 10, figsize=(12, 1.2 * n_show))

for row, si in enumerate(snapshot_idx):
    for col in range(10):
        img = x_gen_store[si, col, 0]
        axes[row, col].imshow(np.clip(img, 0, 1), cmap='gray', vmin=0, vmax=1)
        axes[row, col].axis('off')
    t_approx = n_T - int(si / max(n_snapshots - 1, 1) * n_T)
    axes[row, 0].set_ylabel(f't~{t_approx}', fontsize=9, rotation=0, labelpad=30, va='center')

for col in range(10):
    axes[0, col].set_title(str(col), fontsize=10)

plt.suptitle('Reverse diffusion trajectory: noise (top) -> digit (bottom)', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

print(f'Showing {n_show} snapshots from T={n_T} (pure noise) to t=0 (generated digit)')

### What we just demonstrated

The MNIST DDPM above illustrates exactly the same algorithm used in materials generation:

| MNIST (images) | Materials (crystals) |
|----------------|---------------------|
| Pixel values in [0, 1] | Fractional coordinates in [0, 1) |
| 28x28 grid | Variable number of atoms |
| UNet predicts noise | GNN (CSPNet) predicts noise |
| Class label (digit 0-9) | Structural constraint (kagome, honeycomb) |
| Classifier-free guidance | Inpainting with fixed atom positions |

The key differences for materials:
- Coordinates are **periodic** (wrapped normal distribution)
- Must also diffuse the **lattice** and **atom types**
- Graph neural networks replace convolutional networks

These extensions are covered in Notebook 03 (DiffCSP/materials diffusion).

---
## 2.6 The landscape of crystal generative models

| Model | Year | Type | Key feature |
|-------|------|------|-------------|
| CDVAE | 2022 | VAE | First crystal generative model with graph decoder |
| DiffCSP | 2023 | Diffusion | Crystal structure prediction by joint equivariant diffusion |
| SCIGEN | 2025 | Diffusion | Structural constraints (kagome, honeycomb, etc.) via inpainting |
| MatterGen | 2025 | Diffusion | Property-guided generation |
| UniMat | 2025 | Diffusion | Universal materials generation |

All of these use the same core DDPM ideas we just demonstrated on MNIST —
adapted for 3D periodic structures with graph neural networks.

In the next notebooks, we'll see how DiffCSP and SCIGEN apply diffusion
specifically to crystal structures.

---
## Key takeaways

1. **Diffusion models** (DDPM) generate data by learning to reverse a noising process.
2. The forward process: $q(\mathbf{x}_t | \mathbf{x}_0) = \mathcal{N}(\sqrt{\bar{\alpha}_t}\,\mathbf{x}_0,\; (1-\bar{\alpha}_t)\mathbf{I})$
3. A neural network learns to predict the noise: $\mathcal{L} = \|\boldsymbol{\epsilon} - \boldsymbol{\epsilon}_\theta(\mathbf{x}_t, t)\|^2$
4. **Classifier-free guidance** enables conditional generation (specific digit classes, or specific lattice types).
5. The same DDPM framework extends to crystal structures by replacing images with atom coordinates + lattice matrices.

In the next notebook, we'll see how this works for **materials structure prediction**.

---
## References

- **DDPM:** Ho, Jain & Abbeel, "Denoising Diffusion Probabilistic Models," *NeurIPS* (2020). [arXiv:2006.11239](https://arxiv.org/abs/2006.11239)
- **Classifier-Free Guidance:** Ho & Salimans, "Classifier-Free Diffusion Guidance," *NeurIPS Workshop* (2022). [arXiv:2207.12598](https://arxiv.org/abs/2207.12598)
- **Score matching:** Song & Ermon, "Generative Modeling by Estimating Gradients of the Data Distribution," *NeurIPS* (2019). [arXiv:1907.05600](https://arxiv.org/abs/1907.05600)
- **MNIST DDPM code:** TeaPearce/Conditional_Diffusion_MNIST (MIT License). [GitHub](https://github.com/TeaPearce/Conditional_Diffusion_MNIST)
- **minDiffusion:** cloneofsimo/minDiffusion. [GitHub](https://github.com/cloneofsimo/minDiffusion)
- **CDVAE:** Xie et al., "Crystal Diffusion Variational Autoencoder," *ICLR* (2022). [arXiv:2110.06197](https://arxiv.org/abs/2110.06197)
- **DiffCSP:** Jiao et al., "Crystal Structure Prediction by Joint Equivariant Diffusion," *NeurIPS* (2023). [arXiv:2309.04475](https://arxiv.org/abs/2309.04475)
- **SCIGEN:** Okabe et al., "Structural constraint integration in a generative model for the discovery of quantum materials," *Nature Materials* (2025). [DOI](https://doi.org/10.1038/s41563-025-02355-y)

---
## What's next?

Proceed to **Notebook 03: Diffusion for Materials** — where we apply the DDPM framework
to crystal structures, handling periodic coordinates, lattice matrices, and atom types.